#PySpark Practice

In [0]:
print ("hello world")

#Create DataFrame using Python Objects

In [0]:
# Create a DataFrame from a list of tuples.
#First argument is an array of tuples (Rows) and second is an array of column names 
# Create a DataFrame with column names specified.
spark.createDataFrame([('Alice', 1)], ['name', 'age']).show()
# +-----+---+
# | name|age|
# +-----+---+
# |Alice|  1|
# +-----+---+

df = spark.createDataFrame([('Alice', 14, False), ('Sugar', 13, True)], ['name', 'age'])
#+-----+---+-----+
#| name|age|   _3|
#+-----+---+-----+
#|Alice| 14|false|
#|Sugar| 13| true|
#+-----+---+-----+

#   Notes : This gives the Column Name from the second array in the arguments.
#	        If we miss a column name in the second array argument then the column will _# # is the column number 
#			Also determines the datatype, but using the first value. i.e. if we give 1 and 1.2 it will fail [CANNOT_MERGE_TYPE] Can not merge type `DoubleType` and `LongType`.


#below shows the data in table format 
#df.display()

##
print(df)  #returns the below schema list
#DataFrame[name: string, age: bigint, _3: boolean]

In [0]:
# Create a DataFrame from a list of dictionaries.
spark.createDataFrame([{'name': 'Alice', 'age': 10}, {'name': 'Bob', 'age': 20}])
df.display()
# +---+-----+
# |age| name|
# +---+-----+
# |  1|Alice|
# +---+-----+
#    Notes : This gives the Column Name from the struct 
#  			 Also determines the datatype, but using the first value. i.e. if we give 1 and 1.2 it will fail [CANNOT_MERGE_TYPE] Can not merge type `DoubleType` and `LongType`.


In [0]:
# Create a DataFrame with an explicit schema. Note : We will have to import the "types" shown below 
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, CharType
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("pass", BooleanType(), True)])
df = spark.createDataFrame([('Alice', 1.0, True), ('Sugar',2, False)], schema)
display (df)
#+-----+---+-----+
#| name|age| pass|
#+-----+---+-----+
#|Alice|  1| true|
#|Sugar|  2|false|
#+-----+---+-----+

#   Notes : This gives the Column Name from the schema variable which is created using StructType / StructField.
#			Also casts the Float to IntegerType automatically 
#			To insert a Date use the below 

In [0]:
#Note : createDataFrame cannot infer datatype Date
#For DateType it is not possible to pas a string. We will have to use the Python Datetime module
import datetime as Datetime 
from pyspark.sql.types import StructType, StructField, DateType
schema = StructType([
    StructField("name", DateType(), True)])
  
df = spark.createDataFrame([(Datetime.date(2025,12,12))], schema)
display (df) 

#df = spark.createDataFrame([("2025-12-12")], schema)  #This will fail with below error message 
#ArrowTypeError: object of type <class 'str'> cannot be converted to int
#wsfs/fuse/wsfs.go:964 Error(message=Cannot find child LearnDatabricks, errno=no such file or directory, statusCode=0, stack=<nil>)
#[Trace ID: 00-e88ea90823c9249909a0617f56685e2a-4c51af4c15b29013-00]

In [0]:
# Create a DataFrame with a DDL-formatted schema string.
spark.createDataFrame([('Alice', 1)], "name: string, age: int").show()
# +-----+---+
# | name|age|
# +-----+---+
# |Alice|  1|
# +-----+---+


In [0]:
# Create a DataFrame from Row objects.
from pyspark.sql import Row
Person = Row('name', 'age')
spark.createDataFrame([Person("Alice", 1)]).show()
# +-----+---+
# | name|age|
# +-----+---+
# |Alice|  1|
# +-----+---+		

In [0]:
%python
################################################################################################
#                          Sample dataframe for trying out commands                            #
################################################################################################
import pyspark.sql.types as T

emp_data = [
    [1,101,"John Doe",30,"Male",50000,"2015-01-01"],
    [2,101,"Jane Smith",25,"Female",62000,"2016-02-15"],
    [3,102,"Bob Brown",35,"Male",55000,"2014-05-01"],
    [4,102,"Alice Lee",28,"Female",49000,"2017-09-30"]
]

emp_schema = T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("dept_id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("age", T.IntegerType(), True),
    T.StructField("gender", T.StringType(), True),
    T.StructField("salary", T.DoubleType(), True),
    T.StructField("start_date", T.StringType(), True)  #Note this is StringType and not DateType. If it is DateType we will have to use the Python Datetime module
])

emp = spark.createDataFrame(emp_data, emp_schema)

In [0]:
#to Display in tabular format 
emp.display()

In [0]:
#to show the output in text but tabular format 
emp.show()

In [0]:
#below give the number of rows in the dataframe
emp.count()

#aggregate 
emp.agg({"*": "count"}).show()
#+--------+
#|count(1)|
#+--------+
#|       4|
#+--------+

emp.agg({"salary": "sum"}).show()
#+-----------+
#|sum(salary)|
#+-----------+
#|   216000.0|
#+-----------+

emp.agg({"age": "max"}).show()
#+--------+
#|max(age)|
#+--------+
#|      35|
#+--------+

#for multiple columns
emp.agg({"age": "max", "salary":"sum"}).show()
#+--------+-----------+
#|max(age)|sum(salary)|
#+--------+-----------+
#|      35|   216000.0|
#+--------+-----------+

#Note cannot get multiple agg in the same line. Does not fail but it shows the last one only
emp.agg({"age": "max", "age": "min"}).show()

#To get multiple agg in the same line, use the below whe eill have to use the functions 


In [0]:
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html
import pyspark.sql.functions as F

emp.select(F.max("age"), F.min("age")).show()

In [0]:
#This is like a pointer to the dataframe and can be used when we do self join 
emp_side = emp.alias("emp")
mgr_side = emp.alias("mgr")

In [0]:
#DataFrame.checkpoint() is a powerful mechanism used to truncate the lineage (execution plan) of a DataFrame and save its current state to a reliable, persistent storage system (like HDFS, S3, or ADLS).
#You should incorporate df.checkpoint() into your data pipelines when:
#   You are building loops: You are updating a DataFrame inside a for or while loop repeatedly (common in graph processing, machine learning, or processing multi-level corporate hierarchies).
#   You are encountering StackOverflowError: Your Spark driver is crashing during plan optimization phases due to an unmanageably large DAG.
#   Resilience across jobs is required: You want to save an intermediate state of an incredibly expensive calculation so that if a long-running multi-hour script crashes in Phase 2, you don't have to restart Phase 1 from scratch.



In [0]:
#### DataFrame.cache() should be used when you intend to reuse the exact same DataFrame multiple times in downstream actions within your Spark application.

## 1. Read and perform expensive cleaning
#df_clean = spark.read.parquet("raw_data/").filter(...).join(...)

## 2. Cache it because it's the foundation for multiple downstream steps
#df_clean.cache()

## Action A: Write a summary report
#df_clean.groupBy("Country").count().write.mode("overwrite").parquet("reports/summary/")

## Action B: Calculate top-tier metrics
#total_revenue = df_clean.agg({"Revenue": "sum"}).collect()[0][0]

## Action C: Save specific records
#df_clean.filter(F.col("Status") == "Flagged").write.jdbc(...)

#Without cache(): Spark will execute the read, filter, and join steps three separate times (once for Action A, once for Action B, and once for Action C).
#With cache(): Spark executes the cleaning logic once for Action A, saves it in RAM, and Action B and C pull directly from memory in a fraction of a second.

#unpersist() to Free Up Memory (Memory is the most precious resource in a Spark cluster. 
#If you cache a DataFrame in Phase 1 of your script, and you no longer need it in Phase 4, you should explicitly delete it from RAM to prevent cluster choking:
#df.unpersist() 


In [0]:
import pyspark.sql.functions as F

#.collect() is to get data a row/column or rows into python list
#This is not recommended as it will get all the data into the driver node and can crash the driver node if the data is too big

#forExample if we want the unique dept_id into a list of rows for looping each dept
depts = emp.select("dept_id").distinct().collect()
print(depts)

type(depts)

for row in depts:
    print(row["dept_id"])

#example 2
summary=emp.agg(F.count("*").alias("count"),F.sum("salary").alias("totalSalary")).collect()
print(summary)

print(f"Number of employees {summary[0]["count"]}, Total Salary {summary[0]["totalSalary"]}")

In [0]:
from pyspark.sql import functions as F

cols = emp.columns
type(cols)
print(cols)

for col in cols:
    print(col)

#Imagine you have a data lake table with 200 columns. Regulations mandate that you must drop or mask any column containing sensitive Personally Identifiable Information (PII) before saving it to a public directory, 
# or drop metadata columns created by an ingestion tool (like columns starting with _sys_).
#Instead of typing out all those columns to drop them, you use df.columns to dynamically find them



# Fetch all columns dynamically
all_cols = emp.columns

# Use Python logic to find any column containing "ssn", "password", or "sys"
pii_and_sys_cols = [
    c for c in all_cols 
    if "name" in c.lower() or "salary" in c.lower() or c.startswith("_sys_")
]

# Drop them all dynamically in one pass
df_secure = emp.drop(*pii_and_sys_cols)

print(df_secure.columns)

#or maybe we want to uppercase all coulmns 
all_cols = emp.columns

upper_cols = [c.upper() for c in all_cols]
    
df_upper = emp.toDF(*upper_cols)  #The .toDF() method is a quick, versatile utility used to rename columns or apply a new schema to an existing DataFrame.
print(df_upper.columns)


In [0]:
#Computes basic statistics for numeric and string columns.
#This includes count, mean, stddev, min, and max. If no columns are given, this function computes statistics for all numerical or string columns.

emp.describe().display()

In [0]:
#Removes distinct rows from the DataFrame.

from pyspark.sql import functions as F
import pyspark.sql.types as T

emp_data = [
    [1,"Alice Lee","Male"],
    [2,"Jane Smith","Female"],
    [3,"Bob Brown","Male"],
    [1,"Alice Lee","Male"],
    [3,"Bob Brown","Male"],
    [2,"Jane Smith","Male"]
]

emp_schema = T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("gender", T.StringType(), True)
])

emp = spark.createDataFrame(emp_data, emp_schema)

emp.sort("id").display()
emp.distinct().sort("id").display()

#if we want disctinct values of a column then select before distinct 
emp.select("name").distinct().display()

#if we want disctinct values for multiple columns then select before distinct 
emp.select("name", "gender").distinct().show()

In [0]:
#drop column from a dataframe 

emp.drop("gender").display()

emp.drop("gender","id").display()

a=["gender", "name"]
emp.drop(*a).display()


In [0]:
#the same result as distinct(), distinct() is a synonym for dropDuplicates() without parameter columns 
emp.dropDuplicates().display()

#using arguments. This is useful when you want to drop duplicates based on a subset of columns
#When you pass a subset of columns into dropDuplicates(), Spark groups the rows by those specific columns. For any column not included in that list, Spark applies a "First One Wins" rule.
#The value selected will come from the very first row that Spark's engine physically processes for that group. Because Spark is a distributed computing system, this selection is non-deterministic (unpredictable) by default.
emp.dropDuplicates(["id","name"]).display()

##you can see that whin I use the orderby before dropDuplicates, the result is different
emp.orderBy("gender", ascending=False).dropDuplicates(["id","name"]).display()

In [0]:
from pyspark.sql import Row
df = spark.createDataFrame([
    Row(age=10, height=80.0, name="Alice"),
    Row(age=5, height=float("nan"), name="Bob"),
    Row(age=None, height=None, name="Tom"),
    Row(age=None, height=float("nan"), name=None),
])

#both the below give same rusult. drops rows which have null values in atleast one column
df.na.drop(how="any").display()
df.dropna(how="any").display()


##only if all columns have null values, then only it will drop the row
df.na.drop(how="all").display()
df.dropna(how="all").display()

#threshold means atleast how many columns should have null values to drop the row 
#thresh defines an exact integer requirement: "Keep only the rows that have at least $n$ non-null values."  
#argument how is ignore when we use thresh
df.na.drop(how="any", thresh=2).display()
df.dropna(how="any", thresh=2).display()



In [0]:

#Returns all column names and their data types as a list.
print(emp.dtypes)

In [0]:

df1 = spark.createDataFrame([("a", 1), ("a", 1), ("a", 1), ("a", 2), ("b",  3), ("c", 4)], ["C1", "C2"])
df2 = spark.createDataFrame([("a", 1), ("b", 3)], ["C1", "C2"])   #, ("a", 1), ("a", 1)

#Note : It only removes the first occurance of ("a",1) from df1, if you want the other then repeat the entry in df2
df1.exceptAll(df2).show()

In [0]:
df1 = spark.createDataFrame([("a", 1), ("a", 1), ("a", 1), ("a", 2), ("b",  3), ("c", 4)], ["C1", "C2"])
df2 = spark.createDataFrame([("a", 1), ("b", 3)], ["C1", "C2"])   #, ("a", 1), ("a", 1)

#Note : this will remove all occurance of ("a",1)
df1.subtract(df2).show()

In [0]:
#The exists method provides a way to create a boolean column that checks for the presence of related records in a subquery. When applied within a DataFrame, this method allows you to filter rows based on whether matching records exist in the related dataset. The resulting Column object can be used directly in filtering conditions or as a computed column.

from pyspark.sql import functions as sf
df1 = spark.createDataFrame([("a", 1), ("x", 7), ("y", 9), ("a", 2), ("b",  3), ("c", 4)], ["C1", "C2"])
df2 = spark.createDataFrame([("a", 1), ("b", 3)], ["C1", "C2"])   #, ("a", 1), ("a", 1)

df1.alias("a").where(df2.alias("b").where(sf.col("b.C1") == sf.col("a.C1").outer()).exists()).orderBy("C1").show()

#When you write a nested subquery (a query inside another query), Spark isolates the inner query's execution context. 
#If the inner query needs to look "up" and reference a column from the main table outside itself, you must explicitly tag that column with .outer().
#Without .outer(), Spark's analyzer will only look inside the immediate subquery, fail to find or correctly bind the outer column, and throw an error.

In [0]:
df = spark.createDataFrame(
    [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df.explain()  

df.explain(extended=True)

df.explain(mode="formatted")

#df.explain(extended=True, mode="formatted")     fails with below error 
#[CANNOT_SET_TOGETHER] extended and mode should not be set together.



In [0]:
#based on the datatype of the value provided it populates that value into all columns of that datatype in dataFrame 
df = spark.createDataFrame([
    (10, 80.5, "Alice", "Lee", None),
    (5, None, "Bob", "Smith", False),
    (None, None, "Tom", None, None),
    (None, None, None, None, True)],
    schema=["age", "height", "name", "surname", "bool"])

df.printSchema()

#as "N/A" is string, it will replace NULLS in columns name and surname 
df.fillna("N/A").show()

#this will fill both age and height 
df.fillna(9999).show()

#this will fill both name and surname as the numer is in ""
df.fillna("9999").show()

#this will fill the bool column
df.fillna(True).show()

#although I am specifying subset, it will fill not fill any as the value is in ""
print ('Testing the value in "" for a Integer /')
df.fillna("9999", subset=["age","height"]).show()

#this fills both age & heaight 
df.fillna(99, subset=["age","height"]).show()

#this fills both age & height 
df.fillna({"age":99,"name":"N/A"}).show()

#this fills both age & height. Even if the value is in "" it will fill the column age
df.fillna({"age":"99","name":"N/A"}).show()

In [0]:
#As it returns a row(python object), it is not possible to join with a dataFrame. For joining use .limit(1) which returns a dataFrame
df = spark.createDataFrame([
    (10, 80.5, "Alice", "Lee", None),
    (5, None, "Bob", "Smith", False),
    (None, None, "Tom", None, None),
    (None, None, None, None, True)],
    schema=["age", "height", "name", "surname", "bool"])

row = df.first()
print(row)

row = df.orderBy("name", ascending=False).first()
print(row)


In [0]:
#This is to get top n number of rows. This is like SQL TOP which can be used in join 
df = spark.createDataFrame([
    (10, 80, "Alice", "Lee", None),
    (5, 20, "Bob", "Smith", False),
    (1, 30, "Tom", None, None),
    (3, 19, None, None, True)],
    schema=["age", "height", "name", "surname", "bool"])

df.limit(3).show() 

df.orderBy("age").limit(3).show() 



In [0]:

#In PySpark, DataFrame.foreach() is an action that applies a custom function to every single row in your DataFrame.
#Unlike transformations (such as .map() or .filter()) which return a new DataFrame, foreach() is designed purely for side effects. It processes the data and returns absolutely nothing (None). It is typically used when you need to send your data #outside of Spark—such as writing rows to an external database, sending records to a web API, or triggering alerts.

# Sample DataFrame
df = spark.createDataFrame([
    (101, "Alice", "alice@example.com"),
    (102, "Bob", "bob@example.com")
], ["user_id", "name", "email"])

# 1. Define a function that takes a Row object
def send_welcome_email(row):
    # Access columns using dot-notation or bracket-notation
    user_name = row.name      # or row["name"]
    user_email = row.email    # or row["email"]
    
    # Imagine calling an external email service API here
    print(f"Sending email to {user_name} at {user_email}")

# 2. Pass the function name directly into .foreach()
df.foreach(send_welcome_email)


#foreachPartition() does not send one row at a time to your function. This is the fundamental difference between it and foreach().While foreach() invokes your function $N$ times (once for every single row in the DataFrame), foreachPartition() #invokes your function exactly once per partition.Instead of a single Row object, Spark passes a Python iterator (an iterable collection containing all the rows allocated to that specific partition) into your function.

# If your data has 4 partitions, this function is only called 4 times!
def process_partition(partition_iterator):
    # 'partition_iterator' is a Python iterator containing thousands of rows
    
    # 2. YOU loop through the rows one by one inside the function
    for row in partition_iterator:
       print(row.user_id)
      
df.foreachPartition(process_partition)

#Note : You will see a Warning as it is trying to say that there is no output. these should be used when ypu are transferring the control to outside of spark ex. write to DB, call and API with params etc. 

In [0]:

#A very powerful groupBy which can let you do aggregates for different combinations in one go. 
#This reduces the io as data is being only ready once. 

from pyspark.sql import functions as sf
df = spark.createDataFrame([
    (100, 'Fremont', 'Honda Civic', 10),
    (100, 'Fremont', 'Honda Accord', 15),
    (100, 'Fremont', 'Honda CRV', 7),
    (200, 'Dublin', 'Honda Civic', 20),
    (200, 'Dublin', 'Honda Accord', 10),
    (200, 'Dublin', 'Honda CRV', 3),
    (300, 'San Jose', 'Honda Civic', 5),
    (300, 'San Jose', 'Honda Accord', 8)
], schema="id INT, city STRING, car_model STRING, quantity INT")

print(f"df is Local {df.isLocal()}")

#Arg1 : and array of column combinations to group by
#Arg2 : columns to group by

#and then chain the .agg to get the aggregates 

df.groupingSets(
    [("city", "car_model"), ("city",),("car_model",),("id",)],
    "city", "car_model", "id"
).agg(sf.sum(sf.col("quantity")).alias("sum")).display()



In [0]:
#Returns Python list 
#Calling df.head(5) is syntactic sugar for df.limit(5).collect()

top5rows = df.head(3)

for row in top5rows:
  print(f"{row.quantity} {row.car_model}'a were sold from  {row.city} ")

top5rows = df.sort(sf.desc("quantity")).head(3)

for row in top5rows:
  print(f"{row.quantity} {row.car_model}'a were sold from  {row.city} ")


Leaving the .hint() as I need to understand spark optimisation 

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

populationFilePath = "/Volumes/databrickspractice/landing/datasets/csv/WorldPopulation/WorldPopulation.csv"
regionsFilePath = "/Volumes/databrickspractice/landing/datasets/csv/Regions/Regions.csv"
subRegionsFilePath = "/Volumes/databrickspractice/landing/datasets/csv/SubRegions/SubRegions.csv"

catalog_schema = "databrickspractice.bronze."

populationSchema = StructType([
                                StructField('ISON', IntegerType(), False), 
                                StructField('ISO3', StringType(), False), 
                                StructField('Country', StringType(), False), 
                                StructField('Capital', StringType(), True), 
                                StructField('RegionId', IntegerType(), False), 
                                StructField('SubRegionId', IntegerType(), True), 
                                StructField('Continent', StringType(), True), 
                                StructField('Rank', IntegerType(), True), 
                                StructField('Population2022', IntegerType(), True), 
                                StructField('Population2020', IntegerType(), True), 
                                StructField('Area', IntegerType(), True), 
                                StructField('Density', DoubleType(), True), 
                                StructField('GrowthRate', DoubleType(), True), 
                                StructField('WorldPopulationPercentage', DoubleType(), True)
                              ])

df = spark.read.format("csv").\
    schema(populationSchema).\
    options(header="true").\
    load(populationFilePath)

df.inputFiles()

In [0]:
#to check if the dataFrame is Empty
dfNull = df.filter(df.ISON.isNull())
print(f"Number of rows {dfNull.count()}")

dfNull.isEmpty()

In [0]:
%sql
USE CATALOG databrickspractice;
USE SCHEMA information_schema;

In [0]:
# A metadata query to show tables in the catalog
#Useful for unit testing 
df_tables = spark.sql("SHOW TABLES")

df_tables.display()

print(df_tables.isLocal())

# Output: True

Joins 

In [0]:
import datetime as Datetime 
from pyspark.sql.functions import lit, col, expr, when

df_ins = spark.createDataFrame([
(101,'Entity1', Datetime.date(2020,1,1),  Datetime.date(9999,12,31)),
(102,'Entity2', Datetime.date(2021,1,1),  Datetime.date(9999,12,31)),
(103,'Entity3', Datetime.date(2020,1,1),  Datetime.date(9999,12,31)),
(104,'Entity4', Datetime.date(2022,1,1),  Datetime.date(9999,12,31)),
(105,'Entity5', Datetime.date(2020,1,1),  Datetime.date(2025,12,31)),
(106,'Entity6', Datetime.date(2023,1,1),  Datetime.date(2026,12,31)),
(107,'Entity7', Datetime.date(2020,1,1),  Datetime.date(9999,12,31))
], schema="id INT, name STRING, vFrom DATE, vTo DATE")


df_ids = spark.createDataFrame([
(101,'SEDOL','101SED', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(101,'ISIN' ,'101ISN', Datetime.date(2020,1,1), Datetime.date(2023,12,31)),
(101,'ISIN' ,'111ISN', Datetime.date(2024,1,1), Datetime.date(9999,12,31)),
(101,'CUSIP','101CUP', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(101,'LOCAL','101LOC', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(102,'SEDOL','102SED', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(102,'ISIN' ,'102ISN', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(102,'CUSIP','102CUP', Datetime.date(2020,1,1), Datetime.date(2023,12,31)),
(102,'CUSIP','112CUP', Datetime.date(2024,1,1), Datetime.date(9999,12,31)),
(102,'LOCAL','102LOC', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(103,'SEDOL','103SED', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(103,'ISIN' ,'103ISN', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(103,'CUSIP','103CUP', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(103,'LOCAL','103LOC', Datetime.date(2020,1,1), Datetime.date(2021,12,31)),
(103,'LOCAL','113LOC', Datetime.date(2022,1,1), Datetime.date(9999,12,31)),
(104,'SEDOL','104SED', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(104,'ISIN' ,'104ISN', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(104,'CUSIP','104CUP', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(104,'LOCAL','104LOC', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(105,'SEDOL','105SED', Datetime.date(2020,1,1), Datetime.date(2025,12,31)),
(105,'ISIN' ,'105ISN', Datetime.date(2020,1,1), Datetime.date(2025,12,31)),
(105,'CUSIP','105CUP', Datetime.date(2020,1,1), Datetime.date(2025,12,31)),
(105,'LOCAL','105LOC', Datetime.date(2020,1,1), Datetime.date(2025,12,31)),
(106,'SEDOL','106SED', Datetime.date(2020,1,1), Datetime.date(2026,12,31)),
(106,'ISIN' ,'106ISN', Datetime.date(2020,1,1), Datetime.date(2026,12,31)),
(106,'CUSIP','106CUP', Datetime.date(2020,1,1), Datetime.date(2025,12,31)),
(106,'CUSIP','116CUP', Datetime.date(2026,1,1), Datetime.date(2026,12,31)),
(106,'LOCAL','106LOC', Datetime.date(2020,1,1), Datetime.date(2026,12,31)),
(107,'SEDOL','107SED', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(107,'ISIN' ,'107ISN', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(107,'CUSIP','107CUP', Datetime.date(2020,1,1), Datetime.date(9999,12,31)),
(107,'LOCAL','107LOC', Datetime.date(2020,1,1), Datetime.date(9999,12,31))
], schema="id INT, idtype STRING, idValue STRING, vFrom DATE, vTo DATE")

from pyspark.sql import functions as sf
df_sh = spark.createDataFrame([
(101, Datetime.date(2019,12,1),1001),
(101, Datetime.date(2020,1,5),1002),
(101, Datetime.date(2020,10,1),1003),
(102, Datetime.date(2019,12,1),2002),
(102, Datetime.date(2020,1,5),2003),
(102, Datetime.date(2020,10,1),1002),
(103, Datetime.date(2019,12,1),3001),
(103, Datetime.date(2020,1,5),1003),
(103, Datetime.date(2020,10,1),3002),
(104, Datetime.date(2019,12,1),4001),
(104, Datetime.date(2021,1,5),1001),
(104, Datetime.date(2020,10,1),4002),
(105, Datetime.date(2019,12,1),1005),
(105, Datetime.date(2020,1,5),5001),
(105, Datetime.date(2020,10,1),5002),
(106, Datetime.date(2023,12,1),6001),
(106, Datetime.date(2025,12,5),1006),
(106, Datetime.date(2026,10,1),6002),
(107, Datetime.date(2019,12,1),1007),
(107, Datetime.date(2020,1,5),7001),
(107, Datetime.date(2020,10,1),7002)
], schema="id INT, effDate DATE, share INT")

df_ins.display()
df_ids.display()
df_sh.display()


In [0]:
#Different was of selecting columns 
#sql like, put each column name in "" seperated with, 
df_ins.select("id", "name").display()

#wrap the columns in col function. will result in same but col() lets us use the methods on the column like substr()  most of the methods are useful in where 
df_ins.select(col("id"), col("name")).display()

#note we can use both the sql like and col() in the same select
df_ins.select("id", "name", col("name").substr(3,10)).display()

#belowe where we use df.columnname is also similar to col() and lets us use the methods. 
df_ins.select(df_ins.id, df_ins.name).display()
df_ins.select("id", df_ins.name, df_ins.name.substr(3,10)).display()

#to case the column
df_ins.select(df_ins.id.cast("String"), df_ins.name, df_ins.vFrom.cast("String")).printSchema()
df_ins.select(df_ins.id.cast("String"), df_ins.name, df_ins.vFrom.cast("String")).display() 

#how to alias a column in select. not expr should be imported like col 
df_ins.select(df_ins.id.alias("InstrId"), col("name").alias("EntityName"), expr("vFrom as validFrom")).display()

##expr also return column do the methods can be applied 
df_ins.select(expr("vFrom").alias("validFrom")).display()

df_ins.selectExpr("id", "name as EntityName").display()
df_ins.selectExpr("id", "name as EntityName", "cast(vFrom as String)").printSchema()

from pyspark.sql.functions import upper
#transform() This is for applying a function on the column and return a new column. i.e. these functions take column as parameter and not column value. Good use for UDF 
df_ins.select(df_ins.name.transform(upper).alias("upperName")).display()

from pyspark.sql.types import IntegerType
df_ins.select(df_ins.name.try_cast(IntegerType()).alias("IntegerName")).display()  #This will return NULL's but fail

#.when() similar to case in sql 
from pyspark.sql.functions import when
df_ids.select("*", expr("case when '2026-06-11' between vFrom and vTo then 'Current' else 'Historical' end").alias("Status")).display()

#keep chaining the whens 
df_ids.select("*", when(df_ids.id < 103, "Old").when(df_ids.id.between(103, 105), "Oth").otherwise("New").alias("Status")).display()

#put columns in a variable 
##By creating an array of columns 
cols = ["id", "name"] ##Array of column name. note they are strings. 
df_ins.select(*cols).show()  #*cols means unpack the array of column names. 

cols = [col("id"), col("name").alias("entityName")]
df_ins.select(*cols).show()

cols = [col("id"), col("vFrom").cast("String").alias("strDate")]
df_ins.select(*cols).printSchema()
df_ins.select(*cols).show()

In [0]:
df_ins.withColumn("newCol", lit("new")).display()

df_ins.withColumns({"newCol": lit("new"), "newCol2": lit("new2")}).display()

#If the new column name provided is same as an exiting column then it will remove the existing on from resulting dataFrame
df_ins.withColumn("id", lit("new")).display()

In [0]:
df_ins.withColumnRenamed("id", "newId").display()

df_ins.withColumnsRenamed({"id": "newId", "name": "newName"}).display()

In [0]:
#.filter() and .where() are synonymous 

#sql like. put the condition in ""
df_ins.filter("id = 105").display()

#same as above but using the column name
df_ins.filter(df_ins.id == 105).display()

#multiple conditions. both the below give same result 
df_ins.filter("id = 105 and name = 'Entity5'").display()
df_ins.filter((df_ins.id == 105) & (df_ins.name == "Entity5")).display() 

#multiple conditions. both the below give same result 
#notice both conditions are wrapped in brackets as the & | expects the LHS and RHS to be boolean
df_ins.filter("id = 105 or name = 'Entity6'").display()
df_ins.filter((df_ins.id == 105) | (df_ins.name == "Entity6")).display() 

#not equal 
df_ins.filter("id <> 105").display()
df_ins.filter(df_ins.id != 105).display()

#between. both below give the same result. sf.lit is the function for literal value. lit has to be used for chaining if we don't want to chain then write as the 3rd 
df_ins.filter("'2026-05-05' BETWEEN vFrom AND vTo").display()
df_ins.filter(lit(Datetime.date(2026,5,5)).between(df_ins.vFrom, df_ins.vTo)).display()

filt_date = Datetime.date(2026,5,5)
df_ins.filter((df_ins.vFrom <= filt_date) & (df_ins.vTo >= filt_date)).display()

df_ins.filter((df_ins.vFrom <= Datetime.date(2026,5,5)) & (df_ins.vTo >= Datetime.date(2026,5,5))).display()

# isin  is a column method which returns true or false 
insList = [105, 106]
df_ins.filter(df_ins.id.isin(insList)).display()

#isNull
df_ins.filter(df_ins.id.isNull()).display()

#isNotNull
df_ins.filter(df_ins.id.isNotNull()).display()

#startswith
df_ins.filter(df_ins.name.startswith("Entity")).display()

#endswith
df_ins.filter(df_ins.name.endswith("5")).display()

#ilike  Case Insensitive like SQL 
df_ins.filter(df_ins.name.ilike("%Y5")).display()

#like  Case Sensitive like SQL 
df_ins.filter(df_ins.name.like("%y5")).display()

#rlike like using regex
df_ins.filter(df_ins.name.rlike("^E")).display()

# .contains is similar to like. i.e. does a string exist in the value of the column
df_ins.filter(df_ins.name.contains("5")).display() 
df_ins.withColumn("check", df_ins.name.contains("5")).display()

df_ins.withColumn("check", df_ins.name.like("%5")).display()


In [0]:
from pyspark.sql import Row
#below create a structType column with name r 
df = spark.createDataFrame([Row(r=Row(a=1, b="b"))])
df.printSchema()
#getfield is a column method to get an element from a structType column
df.select(df.r.getField("b")).show()
df.select(df.r.a).show()

In [0]:
#useful for getting an item from an array or map
df = spark.createDataFrame([([1, 2], {"key": "value"})], ["l", "d"])
df.printSchema()
df.display()
df.select(df.l.getItem(0), df.d.getItem("key")).show()


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import lit
df = df = spark.createDataFrame([Row(r=1, a=Row(b=1, c=2)), Row(r=2, a=Row(b=2, c=3))])
df.printSchema()
#root
# |-- r: long (nullable = true)
# |-- a: struct (nullable = true)
# |    |-- b: long (nullable = true)
# |    |-- c: long (nullable = true)

df.show()
#+---+------+
#|  r|     a|
#+---+------+
#|  1|{1, 2}|
#|  2|{2, 3}|
#+---+------+

#as the first parameter of wilthField is d this is not in struct, this replaces the value of b with literal 3
df.withColumn('x', df['a'].withField('d', lit(4))).show()
#+---+------+---------+
#|  r|     a|        x|
#+---+------+---------+
#|  1|{1, 2}|{1, 2, 4}|
#|  2|{2, 3}|{2, 3, 4}|
#+---+------+---------+

#as the first parameter of wilthField is b which is already in struct, this replaces the value of b with literal 3
df_new = df.withColumn('x', df.a.withField('b', lit(3))).show()
#+---+------+------+
#|  r|     a|     x|
#+---+------+------+
#|  1|{1, 2}|{3, 2}|
#|  2|{2, 3}|{3, 3}|
#+---+------+------+

#creating a new column in the struct by adding the field in struct
df_new = df.withColumn('x', df.a.withField('d', df.a.getField("b") + df.a.getField("c"))).show()
#+---+------+---------+
#|  r|     a|        x|
#+---+------+---------+
#|  1|{1, 2}|{1, 2, 3}|
#|  2|{2, 3}|{2, 3, 5}|
#+---+------+---------+

#creating a new column in the struct by adding the field in struct and the other column 
df_new = df.withColumn('x', df.a.withField('d', df.r + df.a.getField("b") + df.a.getField("c"))).show()
#+---+------+---------+
#|  r|     a|        x|
#+---+------+---------+
#|  1|{1, 2}|{1, 2, 4}|
#|  2|{2, 3}|{2, 3, 7}|
#+---+------+---------+


In [0]:
#both orderBy() & sort() are synonymus 

from pyspark.sql.functions import col
df = spark.createDataFrame([
                        (1, 5.1, "A", 'a'),
                        (2, 4.2, "B", 'A'),
                        (3, None, "C", 'b'),
                        (4, 2.4, "D", 'B'),
                        (5, 1.5, "E", '1')
                     ], 
    ['seq', 'pressure', 'letter', 'inletter'])

df.printSchema()
#root
# |-- seq: long (nullable = true)
# |-- pressure: double (nullable = true)
# |-- letter: string (nullable = true)
# |-- inletter: string (nullable = true)

df.orderBy("seq").show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  1|     5.1|     A|       a|
#|  2|     4.2|     B|       A|
#|  3|    NULL|     C|       b|
#|  4|     2.4|     D|       B|
#|  5|     1.5|     E|       1|
#+---+--------+------+--------+

df.orderBy("seq", ascending=False).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  3|    NULL|     C|       b|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#+---+--------+------+--------+

df.orderBy(df.seq.asc()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  1|     5.1|     A|       a|
#|  2|     4.2|     B|       A|
#|  3|    NULL|     C|       b|
#|  4|     2.4|     D|       B|
#|  5|     1.5|     E|       1|
#+---+--------+------+--------+

df.orderBy(df.seq.desc()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  3|    NULL|     C|       b|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#+---+--------+------+--------+

df.orderBy(df.pressure.asc()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  3|    NULL|     C|       b|
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#+---+--------+------+--------+

df.orderBy(df.pressure.asc_nulls_last()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#|  3|    NULL|     C|       b|
#+---+--------+------+--------+

df.orderBy(df.pressure.asc_nulls_first()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  3|    NULL|     C|       b|
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#+---+--------+------+--------+

df.orderBy(df.inletter).show() ##Note in mixed case and numbers  Number, Upper, lower
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  5|     1.5|     E|       1|
#|  2|     4.2|     B|       A|
#|  4|     2.4|     D|       B|
#|  1|     5.1|     A|       a|
#|  3|    NULL|     C|       b|
#+---+--------+------+--------+

df.orderBy(df.letter.desc_nulls_last()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  3|    NULL|     C|       b|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#+---+--------+------+--------+

df.orderBy(df.letter.desc_nulls_first()).show() 
#+---+--------+------+--------+
#|seq|pressure|letter|inletter|
#+---+--------+------+--------+
#|  5|     1.5|     E|       1|
#|  4|     2.4|     D|       B|
#|  3|    NULL|     C|       b|
#|  2|     4.2|     B|       A|
#|  1|     5.1|     A|       a|
#+---+--------+------+--------+
